<a href="https://colab.research.google.com/github/Aniketh78/Generative-AI-Lab_Experiments/blob/main/genAiExp06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

df = pd.read_csv("/content/movies.csv")
print(df)
df = df.dropna()

df["genres"] = df["genres"].str.replace("|", " ")

df["combined"] = df["title"] + " " + df["genres"]

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(df["combined"].tolist(), show_progress_bar=True)

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"Indexed {index.ntotal} movies.")

metadata = df[["id", "title", "genres"]].reset_index(drop=True)

query = input("Enter your search query: ")

query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

faiss.normalize_L2(query_embedding)

k = 5
scores, indices = index.search(query_embedding, k)

print("\n🔍 Top Semantic Results:\n")

for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {metadata.iloc[idx]['title']} ({metadata.iloc[idx]['genres']})")
    print(f"   Similarity Score: {scores[0][i]:.4f}\n")

print("\n🔎 Keyword-Based Results:\n")

keyword_results = df[df["combined"].str.contains(query, case=False, na=False)].head(5)

for i, row in keyword_results.iterrows():
    print(f"- {row['title']} ({row['genres']})")

      index     budget                                    genres  \
0         0  237000000  Action Adventure Fantasy Science Fiction   
1         1  300000000                  Adventure Fantasy Action   
2         2  245000000                    Action Adventure Crime   
3         3  250000000               Action Crime Drama Thriller   
4         4  260000000          Action Adventure Science Fiction   
...     ...        ...                                       ...   
4798   4798     220000                     Action Crime Thriller   
4799   4799       9000                            Comedy Romance   
4800   4800          0             Comedy Drama Romance TV Movie   
4801   4801          0                                       NaN   
4802   4802          0                               Documentary   

                                               homepage      id  \
0                           http://www.avatarmovie.com/   19995   
1          http://disney.go.com/disneypictures/pi

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Indexed 1432 movies.
Enter your search query: Sci-fi movie with aliens

🔍 Top Semantic Results:

1. Alien (Horror Action Thriller Science Fiction)
   Similarity Score: 0.6964

2. Aliens in the Attic (Adventure Comedy Family Fantasy Science Fiction)
   Similarity Score: 0.6384

3. Aliens vs Predator: Requiem (Fantasy Action Science Fiction Thriller Horror)
   Similarity Score: 0.6160

4. Interstellar (Adventure Drama Science Fiction)
   Similarity Score: 0.5201

5. Monsters vs Aliens (Animation Family Adventure Science Fiction)
   Similarity Score: 0.4878


🔎 Keyword-Based Results:

